# 02 — CNN Experiments (PyTorch)
Chest X-Ray Pneumonia Classification

This notebook covers:
1. Re-splitting train/val (official val set is only 16 images)
2. Datasets, transforms, and dataloaders (with class-imbalance handling)
3. Baseline CNN (built from scratch)
4. Transfer learning (ResNet18) as a second experiment
5. Training/evaluation loop with metrics tracked per epoch
6. Comparison, confusion matrix, classification report
7. Saving the best checkpoint for the next pipeline stage

Findings from `01_eda.ipynb` (class imbalance, image mode/size) directly inform the choices below.

## 1. Imports & Config

In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
# ---- Config ----
DATA_DIR = Path("../data/raw/chest_xray")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224          # standard for transfer-learning backbones
BATCH_SIZE = 32
NUM_WORKERS = 2          # bump up if not on Windows main-process constraints
VAL_SPLIT_FROM_TRAIN = 0.15   # re-split since official val/ has only 16 images
NUM_CLASSES = 2
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]

CHECKPOINT_DIR = Path("../models")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Fix the train/val split

The official Kaggle `val/` folder only has 16 images, which is unreliable for model
selection. We instead carve a proper validation set out of `train/`, stratified by class,
and leave `test/` untouched as the final held-out set.

In [ ]:
from sklearn.model_selection import train_test_split
import shutil

RAW_TRAIN = DATA_DIR / "train"
NEW_TRAIN = PROCESSED_DIR / "train"
NEW_VAL = PROCESSED_DIR / "val"

def build_resplit(force=False):
    if NEW_TRAIN.exists() and not force:
        print("Re-split already exists, skipping. Set force=True to rebuild.")
        return

    if NEW_TRAIN.exists():
        shutil.rmtree(NEW_TRAIN)
    if NEW_VAL.exists():
        shutil.rmtree(NEW_VAL)

    for cls in CLASS_NAMES:
        files = list((RAW_TRAIN / cls).glob("*.jpeg")) + list((RAW_TRAIN / cls).glob("*.jpg"))
        train_files, val_files = train_test_split(
            files, test_size=VAL_SPLIT_FROM_TRAIN, random_state=SEED
        )
        (NEW_TRAIN / cls).mkdir(parents=True, exist_ok=True)
        (NEW_VAL / cls).mkdir(parents=True, exist_ok=True)

        for f in train_files:
            shutil.copy(f, NEW_TRAIN / cls / f.name)
        for f in val_files:
            shutil.copy(f, NEW_VAL / cls / f.name)

        print(f"{cls}: {len(train_files)} train, {len(val_files)} val")

build_resplit(force=False)

TEST_DIR = DATA_DIR / "test"   # untouched, final evaluation only

## 3. Transforms & Datasets

Grayscale X-rays are replicated to 3 channels so pretrained ImageNet backbones (which
expect 3-channel input) can be used later. Augmentation is light — rotations/flips that
make anatomical sense; we avoid heavy color jitter since these are grayscale medical images.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(NEW_TRAIN, transform=train_transforms)
val_dataset = datasets.ImageFolder(NEW_VAL, transform=eval_transforms)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transforms)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

### Handle class imbalance

Two options: a `WeightedRandomSampler` (resamples minority class more often) or class
weights inside the loss function. We use a weighted sampler for training data here.

In [ ]:
targets = [label for _, label in train_dataset.samples]
class_counts = np.bincount(targets)
print("Class counts (train):", dict(zip(train_dataset.classes, class_counts)))

class_weights = 1.0 / class_counts
sample_weights = [class_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

### Sanity check — visualize a batch

In [ ]:
def denormalize(img_tensor):
    img = img_tensor.numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i, ax in enumerate(axes):
    ax.imshow(denormalize(images[i]))
    ax.set_title(train_dataset.classes[labels[i]])
    ax.axis("off")
plt.show()

## 4. Experiment 1 — Baseline CNN (from scratch)

A simple convolutional network to establish a baseline before trying transfer learning.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                      # 224 -> 112

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                      # 112 -> 56

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                      # 56 -> 28

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                      # 28 -> 14
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

baseline_model = BaselineCNN(NUM_CLASSES).to(DEVICE)
print(baseline_model)

## 5. Reusable training & evaluation loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return running_loss / total, correct / total, all_labels, all_preds, all_probs


def run_experiment(model, train_loader, val_loader, num_epochs=10, lr=1e-4, weight_decay=1e-4,
                    experiment_name="experiment"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"[{experiment_name}] Epoch {epoch}/{num_epochs} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, history

## 6. Run Experiment 1 — Baseline CNN

In [ ]:
baseline_model, baseline_history = run_experiment(
    baseline_model, train_loader, val_loader,
    num_epochs=10, lr=1e-3, weight_decay=1e-4,
    experiment_name="baseline_cnn"
)

## 7. Experiment 2 — Transfer Learning (ResNet18)

Same data, same training loop — only the model changes. Freezing early layers first, then
you can unfreeze for fine-tuning if results look promising.

In [ ]:
def build_resnet18(num_classes=2, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    # Replace final layer — always trainable
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

resnet_model = build_resnet18(NUM_CLASSES, freeze_backbone=True)

In [ ]:
resnet_model, resnet_history = run_experiment(
    resnet_model, train_loader, val_loader,
    num_epochs=8, lr=1e-3, weight_decay=1e-4,
    experiment_name="resnet18_frozen"
)

### Optional: fine-tune (unfreeze) ResNet18 for a few more epochs at a lower LR

Run this cell only if the frozen-backbone result looks promising and you want to squeeze
out more performance.

In [ ]:
for param in resnet_model.parameters():
    param.requires_grad = True

resnet_model, resnet_finetune_history = run_experiment(
    resnet_model, train_loader, val_loader,
    num_epochs=5, lr=1e-5, weight_decay=1e-4,
    experiment_name="resnet18_finetuned"
)

## 8. Compare training curves

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title(f"{title} - Loss")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title(f"{title} - Accuracy")
    axes[1].legend()
    plt.show()

plot_history(baseline_history, "Baseline CNN")
plot_history(resnet_history, "ResNet18 (frozen)")

## 9. Final evaluation on the held-out test set

Pick whichever model performed best on validation, then evaluate once on `test/`.
Reporting more than accuracy matters here — for pneumonia screening, **recall** (catching
actual pneumonia cases) is usually prioritized over raw accuracy.

In [ ]:
def full_test_report(model, test_loader, model_name="model"):
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, y_true, y_pred, y_prob = evaluate(model, test_loader, criterion, DEVICE)

    print(f"=== {model_name} — Test Results ===")
    print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}\n")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(cmap="Blues")
    plt.title(f"{model_name} — Confusion Matrix")
    plt.show()

    return {"test_loss": test_loss, "test_acc": test_acc,
            "roc_auc": roc_auc_score(y_true, y_prob)}

baseline_results = full_test_report(baseline_model, test_loader, "Baseline CNN")
resnet_results = full_test_report(resnet_model, test_loader, "ResNet18")

In [ ]:
comparison = pd.DataFrame([
    {"model": "Baseline CNN", **baseline_results},
    {"model": "ResNet18", **resnet_results},
])
comparison

## 10. Save the best checkpoint

Whichever experiment wins goes forward into the formal pipeline (MLflow tracking,
Model Registry). Save both the weights and the config needed to reconstruct the model.

In [ ]:
best_model, best_name = (
    (resnet_model, "resnet18") if resnet_results["roc_auc"] >= baseline_results["roc_auc"]
    else (baseline_model, "baseline_cnn")
)

checkpoint_path = CHECKPOINT_DIR / f"{best_name}_best.pth"
torch.save({
    "model_state_dict": best_model.state_dict(),
    "model_name": best_name,
    "img_size": IMG_SIZE,
    "class_names": CLASS_NAMES,
    "normalize_mean": IMAGENET_MEAN,
    "normalize_std": IMAGENET_STD,
}, checkpoint_path)

print(f"Saved best model ({best_name}) to {checkpoint_path}")

---
### Next steps

- Wrap this training loop into `src/components/model_trainer.py` for the formal pipeline
- Log each experiment's params/metrics to MLflow instead of printing (next pipeline stage)
- Move the re-split logic into `src/components/data_ingestion.py`
- Move transforms/preprocessing into `src/components/data_preprocessing.py`